# 02b: Feature Engineering

**Purpose:** Transform and encode features for modeling

**Dataset:** COMPAS (cleaned)

**Date:** 2025-11-08

---

## Overview

### Purpose
- Log transformation for skewed count features
- Standardization for continuous features
- One-hot encoding for categorical variables
- Create final feature matrix

### Inputs
- Cleaned data from `02a_data_cleaning.ipynb`

### Outputs
- Transformed features → `data/processed/compas_features_transformed.parquet`
- Transformation metadata → `data/metadata/feature_engineering_log.json`
- Feature names → `data/metadata/feature_names.json`

### Runtime: 1-2 minutes

---

In [2]:
# Setup
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib

project_root = Path.cwd().parent.parent

PROCESSED_DIR = project_root / "data" / "processed"
METADATA_DIR = project_root / "data" / "metadata"

for d in [PROCESSED_DIR, METADATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Setup complete")

✓ Setup complete


## 1. Load Cleaned Data

In [3]:
# Load data components
df_features = pd.read_parquet(PROCESSED_DIR / "compas_features.parquet")
df_target = pd.read_parquet(PROCESSED_DIR / "compas_target.parquet")
df_sensitive = pd.read_parquet(PROCESSED_DIR / "compas_sensitive.parquet")

print(f"Features: {df_features.shape}")
print(f"Target: {df_target.shape}")
print(f"Sensitive: {df_sensitive.shape}")

# Identify feature types
continuous_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_features.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"\nContinuous features: {len(continuous_cols)}")
print(f"Categorical features: {len(categorical_cols)}")

Features: (5209, 8)
Target: (5209, 1)
Sensitive: (5209, 3)

Continuous features: 6
Categorical features: 2


## 2. Log Transformation

Apply log(x + 1) to count features (right-skewed distributions).

In [4]:
# Identify count features
count_features = [col for col in continuous_cols 
                  if 'count' in col.lower() or 'priors' in col.lower()]

print(f"Count features to transform: {count_features}")

# Apply log(x + 1) transformation
df_transformed = df_features.copy()

for col in count_features:
    if col in df_transformed.columns:
        df_transformed[f"{col}_log"] = np.log1p(df_transformed[col])
        print(f"  Transformed: {col} → {col}_log")

# Remove original count features (keep log versions)
df_transformed = df_transformed.drop(columns=count_features)

# Update continuous cols
continuous_cols = df_transformed.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nContinuous features after transform: {len(continuous_cols)}")

Count features to transform: ['juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']
  Transformed: juv_fel_count → juv_fel_count_log
  Transformed: juv_misd_count → juv_misd_count_log
  Transformed: juv_other_count → juv_other_count_log
  Transformed: priors_count → priors_count_log

Continuous features after transform: 6


## 3. Standardization

Standardize continuous features (mean=0, std=1) for logistic regression.

In [7]:
# Fit scaler on continuous features
scaler = StandardScaler()

if len(continuous_cols) > 0:
    df_transformed[continuous_cols] = scaler.fit_transform(df_transformed[continuous_cols])
    print(f"✓ Standardized {len(continuous_cols)} continuous features")
    
    # Save scaler
    joblib.dump(scaler, PROCESSED_DIR / "scaler.joblib")
    print("✓ Saved scaler: scaler.joblib")
else:
    print("No continuous features to standardize")

✓ Standardized 6 continuous features
✓ Saved scaler: scaler.joblib


## 4. One-Hot Encoding

Encode categorical variables (if any in features).

In [8]:
# Check for categorical features in df_features (not in sensitive attributes)
categorical_cols = df_transformed.select_dtypes(exclude=[np.number]).columns.tolist()

if len(categorical_cols) > 0:
    print(f"Encoding {len(categorical_cols)} categorical features: {categorical_cols}")
    
    # One-hot encode
    encoder = OneHotEncoder(drop='first', sparse_output=False)
    encoded = encoder.fit_transform(df_transformed[categorical_cols])
    
    # Create feature names
    encoded_feature_names = encoder.get_feature_names_out(categorical_cols)
    
    # Create DataFrame
    df_encoded = pd.DataFrame(
        encoded,
        columns=encoded_feature_names,
        index=df_transformed.index
    )
    
    # Combine with continuous features
    df_transformed = pd.concat([
        df_transformed.drop(columns=categorical_cols),
        df_encoded
    ], axis=1)
    
    # Save encoder
    joblib.dump(encoder, PROCESSED_DIR / "encoder.joblib")
    print(f"✓ Encoded to {len(encoded_feature_names)} features")
    print("✓ Saved encoder: encoder.joblib")
else:
    print("No categorical features to encode (demographics are in sensitive attributes)")

Encoding 2 categorical features: ['c_charge_degree', 'score_text']
✓ Encoded to 3 features
✓ Saved encoder: encoder.joblib


## 5. Final Feature Matrix

In [9]:
# Final feature matrix
print("Final feature matrix:")
print(f"  Shape: {df_transformed.shape}")
print(f"  Features: {list(df_transformed.columns)}")

# Verify no missing values
assert df_transformed.isnull().sum().sum() == 0, "Transformed data has missing values!"
print("\n✓ No missing values")

# Verify same number of samples
assert len(df_transformed) == len(df_target), "Sample count mismatch!"
print("✓ Sample count matches")

Final feature matrix:
  Shape: (5209, 9)
  Features: ['age', 'decile_score', 'juv_fel_count_log', 'juv_misd_count_log', 'juv_other_count_log', 'priors_count_log', 'c_charge_degree_M', 'score_text_Low', 'score_text_Medium']

✓ No missing values
✓ Sample count matches


## 6. Save Transformed Data

In [10]:
# Save transformed features
output_path = PROCESSED_DIR / "compas_features_transformed.parquet"
df_transformed.to_parquet(output_path, index=False)
print(f"✓ Saved: compas_features_transformed.parquet")

# Save feature names
feature_names = {
    'all_features': list(df_transformed.columns),
    'n_features': len(df_transformed.columns),
    'continuous_features': [col for col in df_transformed.columns 
                           if not any(cat in col for cat in ['race_', 'sex_', 'age_cat_'])],
    'encoded_features': [col for col in df_transformed.columns 
                        if any(cat in col for cat in ['race_', 'sex_', 'age_cat_'])]
}

with open(METADATA_DIR / "feature_names.json", 'w') as f:
    json.dump(feature_names, f, indent=2)
print("✓ Saved: feature_names.json")

# Save engineering log
engineering_log = {
    'notebook': '02b_feature_engineering.ipynb',
    'transformations': {
        'log_transform': count_features,
        'standardization': continuous_cols[:5] if len(continuous_cols) > 5 else continuous_cols,  # sample
        'one_hot_encoding': categorical_cols
    },
    'input_features': len(df_features.columns),
    'output_features': len(df_transformed.columns),
    'n_samples': len(df_transformed),
    'artifacts': {
        'scaler': 'data/processed/scaler.joblib',
        'encoder': 'data/processed/encoder.joblib' if len(categorical_cols) > 0 else None,
        'features': 'data/processed/compas_features_transformed.parquet'
    }
}

with open(METADATA_DIR / "feature_engineering_log.json", 'w') as f:
    json.dump(engineering_log, f, indent=2)
print("✓ Saved: feature_engineering_log.json")

✓ Saved: compas_features_transformed.parquet
✓ Saved: feature_names.json
✓ Saved: feature_engineering_log.json


## Summary

**Feature Engineering Complete:**
- ✓ Log transformation for count features
- ✓ Standardization (mean=0, std=1)
- ✓ One-hot encoding (if categorical)
- ✓ Saved transformers (scaler, encoder)
- ✓ Final feature matrix ready for modeling

**Outputs:**
- compas_features_transformed.parquet
- scaler.joblib (for inverse transform)
- encoder.joblib (if applicable)
- feature_names.json (feature list)
- feature_engineering_log.json (metadata)

**Next:** 02c_train_test_split.ipynb